In [3]:
# TP1 — Semana 3
# Baseline de Projeto de Pesquisa
# RSNA Pediatric Bone Age

# Objetivo:
# Expandir o baseline da Semana 2 com novas famílias de descritores,
# comparar descritores e modelos clássicos e realizar análises
# complementares dos resultados.

print("Semana 3 — expansão do baseline")

Semana 3 — expansão do baseline


In [4]:
# TP1 — Semana 3
# Baseline de Projeto de Pesquisa
# RSNA Pediatric Bone Age

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Ambiente configurado.")
print(f"SEED definida como: {SEED}")

Ambiente configurado.
SEED definida como: 42


In [5]:
# Carregamento do dataset

!pip install -q kagglehub

import kagglehub

dataset_path = kagglehub.dataset_download("kmader/rsna-bone-age")

print("Dataset localizado em:")
print(dataset_path)

Using Colab cache for faster access to the 'rsna-bone-age' dataset.
Dataset localizado em:
/kaggle/input/rsna-bone-age


In [6]:
# Caminhos do dataset

BASE_DIR = dataset_path

TRAIN_CSV = os.path.join(BASE_DIR, "boneage-training-dataset.csv")
TRAIN_IMG_DIR = os.path.join(BASE_DIR, "boneage-training-dataset")

print("CSV encontrado:", os.path.exists(TRAIN_CSV))
print("Pasta de imagens encontrada:", os.path.exists(TRAIN_IMG_DIR))

df = pd.read_csv(TRAIN_CSV)

print("\nDataset carregado!")
print("Quantidade de exames:", len(df))
print("Colunas:", list(df.columns))

CSV encontrado: True
Pasta de imagens encontrada: True

Dataset carregado!
Quantidade de exames: 12611
Colunas: ['id', 'boneage', 'male']


In [7]:
# Recuperação da partição treino/validação

from sklearn.model_selection import train_test_split

df_treino, df_validacao = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED
)

print("Partição recuperada!")
print("Treino:", len(df_treino))
print("Validação:", len(df_validacao))

Partição recuperada!
Treino: 10088
Validação: 2523


In [8]:
# Amostra utilizada nos experimentos

df_treino_amostra = df_treino.sample(
    n=3000,
    random_state=SEED
).copy()

df_validacao_amostra = df_validacao.sample(
    n=750,
    random_state=SEED
).copy()

print("Amostra de treino:", len(df_treino_amostra))
print("Amostra de validação:", len(df_validacao_amostra))
print(
    "Proporção validação:",
    len(df_validacao_amostra) / len(df_treino_amostra)
)

Amostra de treino: 3000
Amostra de validação: 750
Proporção validação: 0.25


In [9]:
# Biblioteca para leitura das imagens

from PIL import Image

print("PIL configurado.")

PIL configurado.


In [10]:
# Preparação do descritor GLCM

from skimage.feature import graycomatrix, graycoprops

def extrair_glcm(imagem_id):
    caminho_imagem = os.path.join(
        TRAIN_IMG_DIR,
        "boneage-training-dataset",
        f"{int(imagem_id)}.png"
    )

    imagem = Image.open(caminho_imagem).convert("L")
    imagem = imagem.resize((256, 256))

    imagem_array = np.array(imagem)

    # Redução dos níveis de cinza para tornar o GLCM mais compacto
    imagem_quantizada = (imagem_array / 256 * 32).astype(np.uint8)

    # Geração da matriz de coocorrência
    glcm = graycomatrix(
        imagem_quantizada,
        distances=[1],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=32,
        symmetric=True,
        normed=True
    )

    # Extração das propriedades de textura
    caracteristicas = []

    propriedades = [
        "contrast",
        "dissimilarity",
        "homogeneity",
        "energy",
        "correlation",
        "ASM"
    ]

    for propriedade in propriedades:
        valores = graycoprops(glcm, propriedade)
        caracteristicas.extend(valores.flatten())

    return np.array(caracteristicas, dtype=np.float32)

In [11]:
# Teste do GLCM em uma única imagem

id_teste = df_treino_amostra.iloc[0]["id"]

caracteristicas_glcm = extrair_glcm(id_teste)

print("ID do exame:", int(id_teste))
print("Quantidade de características:", len(caracteristicas_glcm))
print("Tipo:", caracteristicas_glcm.dtype)
print("Primeiras características:")
print(caracteristicas_glcm[:10])

ID do exame: 6542
Quantidade de características: 24
Tipo: float32
Primeiras características:
[1.3927083  2.1236756  1.0039062  2.1356554  0.39822304 0.4935025
 0.2636489  0.46688196 0.85556626 0.8327812 ]


In [12]:
# Teste de extração GLCM em lote

ids_teste = df_treino_amostra["id"].head(20)

X_glcm_teste = np.array(
    [extrair_glcm(imagem_id) for imagem_id in ids_teste],
    dtype=np.float32
)

print("Formato da matriz:", X_glcm_teste.shape)

Formato da matriz: (20, 24)


In [13]:
# Extração completa das características GLCM — treino

import time

inicio = time.time()

X_treino_glcm = np.array(
    [extrair_glcm(imagem_id) for imagem_id in df_treino_amostra["id"]],
    dtype=np.float32
)

tempo_total = time.time() - inicio

print("Extração GLCM — treino concluída!")
print("Formato:", X_treino_glcm.shape)
print(f"Tempo total: {tempo_total / 60:.2f} minutos")
print(f"Tempo médio por imagem: {tempo_total / len(df_treino_amostra):.3f} segundos")

Extração GLCM — treino concluída!
Formato: (3000, 24)
Tempo total: 3.41 minutos
Tempo médio por imagem: 0.068 segundos


In [14]:
# Salvamento das características GLCM — treino

CAMINHO_GLCM_TREINO = "/content/X_treino_glcm.npy"

np.save(CAMINHO_GLCM_TREINO, X_treino_glcm)

print("GLCM do treino salvo em:")
print(CAMINHO_GLCM_TREINO)

GLCM do treino salvo em:
/content/X_treino_glcm.npy


In [15]:
# Extração completa das características GLCM — validação

inicio = time.time()

X_validacao_glcm = np.array(
    [extrair_glcm(imagem_id) for imagem_id in df_validacao_amostra["id"]],
    dtype=np.float32
)

tempo_total = time.time() - inicio

print("Extração GLCM — validação concluída!")
print("Formato:", X_validacao_glcm.shape)
print(f"Tempo total: {tempo_total / 60:.2f} minutos")
print(f"Tempo médio por imagem: {tempo_total / len(df_validacao_amostra):.3f} segundos")

Extração GLCM — validação concluída!
Formato: (750, 24)
Tempo total: 0.81 minutos
Tempo médio por imagem: 0.064 segundos


In [16]:
# Salvamento das características GLCM — validação

CAMINHO_GLCM_VALIDACAO = "/content/X_validacao_glcm.npy"

np.save(CAMINHO_GLCM_VALIDACAO, X_validacao_glcm)

print("GLCM da validação salvo em:")
print(CAMINHO_GLCM_VALIDACAO)

GLCM da validação salvo em:
/content/X_validacao_glcm.npy


In [17]:
# Preparação das idades ósseas

y_treino = df_treino_amostra["boneage"].values.astype(np.float32)
y_validacao = df_validacao_amostra["boneage"].values.astype(np.float32)

print("y_treino:", y_treino.shape)
print("y_validacao:", y_validacao.shape)

y_treino: (3000,)
y_validacao: (750,)


In [18]:
# Modelo Ridge utilizando características GLCM

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

modelo_ridge_glcm = Ridge(alpha=10.0)

modelo_ridge_glcm.fit(X_treino_glcm, y_treino)

previsoes_ridge_glcm = modelo_ridge_glcm.predict(X_validacao_glcm)

mae_ridge_glcm = mean_absolute_error(
    y_validacao,
    previsoes_ridge_glcm
)

rmse_ridge_glcm = np.sqrt(
    mean_squared_error(
        y_validacao,
        previsoes_ridge_glcm
    )
)

r2_ridge_glcm = r2_score(
    y_validacao,
    previsoes_ridge_glcm
)

print("Ridge + GLCM")
print(f"MAE: {mae_ridge_glcm:.2f} meses")
print(f"RMSE: {rmse_ridge_glcm:.2f} meses")
print(f"R²: {r2_ridge_glcm:.4f}")

Ridge + GLCM
MAE: 32.61 meses
RMSE: 39.56 meses
R²: 0.0724


In [19]:
# Preparação do descritor LBP

from skimage.feature import local_binary_pattern

print("LBP configurado.")

LBP configurado.


In [20]:
# Função para extração das características LBP

def extrair_lbp(imagem_id):
    caminho_imagem = os.path.join(
        TRAIN_IMG_DIR,
        "boneage-training-dataset",
        f"{int(imagem_id)}.png"
    )

    imagem = Image.open(caminho_imagem).convert("L")
    imagem = imagem.resize((256, 256))

    imagem_array = np.array(imagem)

    # Parâmetros do LBP
    pontos = 8
    raio = 1

    lbp = local_binary_pattern(
        imagem_array,
        P=pontos,
        R=raio,
        method="uniform"
    )

    # Histograma dos padrões LBP
    numero_bins = pontos + 2

    histograma, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, numero_bins + 1),
        range=(0, numero_bins)
    )

    # Normalização do histograma
    histograma = histograma.astype(np.float32)
    histograma /= histograma.sum()

    return histograma

In [21]:
# Teste do LBP em uma única imagem

id_teste = df_treino_amostra.iloc[0]["id"]

caracteristicas_lbp = extrair_lbp(id_teste)

print("ID do exame:", int(id_teste))
print("Quantidade de características:", len(caracteristicas_lbp))
print("Tipo:", caracteristicas_lbp.dtype)
print("Histograma LBP:")
print(caracteristicas_lbp)
print("Soma do histograma:", caracteristicas_lbp.sum())

ID do exame: 6542
Quantidade de características: 10
Tipo: float32
Histograma LBP:
[0.0213623  0.04855347 0.01338196 0.08197021 0.14779663 0.15855408
 0.0297699  0.08171082 0.33920288 0.07769775]
Soma do histograma: 1.0


In [22]:
# Teste de extração LBP em lote

ids_teste = df_treino_amostra["id"].head(20)

X_lbp_teste = np.array(
    [extrair_lbp(imagem_id) for imagem_id in ids_teste],
    dtype=np.float32
)

print("Formato da matriz:", X_lbp_teste.shape)

Formato da matriz: (20, 10)


In [23]:
# Extração completa das características LBP — treino

inicio = time.time()

X_treino_lbp = np.array(
    [extrair_lbp(imagem_id) for imagem_id in df_treino_amostra["id"]],
    dtype=np.float32
)

tempo_total = time.time() - inicio

print("Extração LBP — treino concluída!")
print("Formato:", X_treino_lbp.shape)
print(f"Tempo total: {tempo_total / 60:.2f} minutos")
print(f"Tempo médio por imagem: {tempo_total / len(df_treino_amostra):.3f} segundos")


Extração LBP — treino concluída!
Formato: (3000, 10)
Tempo total: 3.19 minutos
Tempo médio por imagem: 0.064 segundos


In [24]:
# Salvamento das características LBP — treino

CAMINHO_LBP_TREINO = "/content/X_treino_lbp.npy"

np.save(CAMINHO_LBP_TREINO, X_treino_lbp)

print("LBP do treino salvo em:")
print(CAMINHO_LBP_TREINO)

LBP do treino salvo em:
/content/X_treino_lbp.npy


In [25]:
# Extração completa das características LBP — validação

inicio = time.time()

X_validacao_lbp = np.array(
    [extrair_lbp(imagem_id) for imagem_id in df_validacao_amostra["id"]],
    dtype=np.float32
)

tempo_total = time.time() - inicio

print("Extração LBP — validação concluída!")
print("Formato:", X_validacao_lbp.shape)
print(f"Tempo total: {tempo_total / 60:.2f} minutos")
print(f"Tempo médio por imagem: {tempo_total / len(df_validacao_amostra):.3f} segundos")

Extração LBP — validação concluída!
Formato: (750, 10)
Tempo total: 0.77 minutos
Tempo médio por imagem: 0.062 segundos


In [26]:
# Salvamento das características LBP — validação

CAMINHO_LBP_VALIDACAO = "/content/X_validacao_lbp.npy"

np.save(CAMINHO_LBP_VALIDACAO, X_validacao_lbp)

print("LBP da validação salvo em:")
print(CAMINHO_LBP_VALIDACAO)

LBP da validação salvo em:
/content/X_validacao_lbp.npy


In [27]:
# Modelo Ridge utilizando características LBP

modelo_ridge_lbp = Ridge(alpha=10.0)

modelo_ridge_lbp.fit(X_treino_lbp, y_treino)

previsoes_ridge_lbp = modelo_ridge_lbp.predict(X_validacao_lbp)

mae_ridge_lbp = mean_absolute_error(
    y_validacao,
    previsoes_ridge_lbp
)

rmse_ridge_lbp = np.sqrt(
    mean_squared_error(
        y_validacao,
        previsoes_ridge_lbp
    )
)

r2_ridge_lbp = r2_score(
    y_validacao,
    previsoes_ridge_lbp
)

print("Ridge + LBP")
print(f"MAE: {mae_ridge_lbp:.2f} meses")
print(f"RMSE: {rmse_ridge_lbp:.2f} meses")
print(f"R²: {r2_ridge_lbp:.4f}")

Ridge + LBP
MAE: 33.17 meses
RMSE: 40.32 meses
R²: 0.0366


In [28]:
# Preparação do descritor HOG

from skimage.feature import hog

TAMANHO = (256, 256)

def extrair_hog_compacto(imagem_id):
    caminho_imagem = os.path.join(
        TRAIN_IMG_DIR,
        "boneage-training-dataset",
        f"{int(imagem_id)}.png"
    )

    imagem = Image.open(caminho_imagem).convert("L")
    imagem = imagem.resize(TAMANHO)

    imagem_array = np.array(imagem)
    imagem_normalizada = imagem_array.astype(np.float32) / 255.0

    caracteristicas = hog(
        imagem_normalizada,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm="L2-Hys"
    )

    return caracteristicas.astype(np.float32)

print("HOG configurado.")

HOG configurado.


In [29]:
# Extração HOG — treino

inicio = time.time()

X_treino_hog = np.array(
    [extrair_hog_compacto(imagem_id) for imagem_id in df_treino_amostra["id"]],
    dtype=np.float32
)

tempo_total = time.time() - inicio

print("Extração HOG — treino concluída!")
print("Formato:", X_treino_hog.shape)
print(f"Tempo total: {tempo_total / 60:.2f} minutos")
print(f"Tempo médio por imagem: {tempo_total / len(df_treino_amostra):.3f} segundos")

Extração HOG — treino concluída!
Formato: (3000, 8100)
Tempo total: 2.97 minutos
Tempo médio por imagem: 0.059 segundos


In [30]:
# Extração HOG — validação

inicio = time.time()

X_validacao_hog = np.array(
    [extrair_hog_compacto(imagem_id) for imagem_id in df_validacao_amostra["id"]],
    dtype=np.float32
)

tempo_total = time.time() - inicio

print("Extração HOG — validação concluída!")
print("Formato:", X_validacao_hog.shape)
print(f"Tempo total: {tempo_total / 60:.2f} minutos")
print(f"Tempo médio por imagem: {tempo_total / len(df_validacao_amostra):.3f} segundos")

Extração HOG — validação concluída!
Formato: (750, 8100)
Tempo total: 0.77 minutos
Tempo médio por imagem: 0.062 segundos


In [31]:
# Salvamento das características HOG — Semana 3

CAMINHO_HOG_TREINO = "/content/X_treino_hog.npy"
CAMINHO_HOG_VALIDACAO = "/content/X_validacao_hog.npy"

np.save(CAMINHO_HOG_TREINO, X_treino_hog)
np.save(CAMINHO_HOG_VALIDACAO, X_validacao_hog)

print("HOG salvo com sucesso!")
print("Treino:", CAMINHO_HOG_TREINO)
print("Validação:", CAMINHO_HOG_VALIDACAO)

HOG salvo com sucesso!
Treino: /content/X_treino_hog.npy
Validação: /content/X_validacao_hog.npy


In [32]:
# Combinação dos descritores HOG + GLCM

X_treino_hog_glcm = np.hstack([
    X_treino_hog,
    X_treino_glcm
])

X_validacao_hog_glcm = np.hstack([
    X_validacao_hog,
    X_validacao_glcm
])

print("HOG + GLCM")
print("Treino:", X_treino_hog_glcm.shape)
print("Validação:", X_validacao_hog_glcm.shape)

HOG + GLCM
Treino: (3000, 8124)
Validação: (750, 8124)


In [33]:
# Recuperação das características HOG da Semana 2

CAMINHO_HOG_TREINO = "/content/X_treino_hog.npy"
CAMINHO_HOG_VALIDACAO = "/content/X_validacao_hog.npy"

X_treino_hog = np.load(CAMINHO_HOG_TREINO)
X_validacao_hog = np.load(CAMINHO_HOG_VALIDACAO)

print("HOG recuperado!")
print("Treino:", X_treino_hog.shape)
print("Validação:", X_validacao_hog.shape)

HOG recuperado!
Treino: (3000, 8100)
Validação: (750, 8100)


In [34]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Pipeline: padronização + Ridge
modelo_ridge_hog_glcm = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=10.0))
])

# Treinamento
modelo_ridge_hog_glcm.fit(
    X_treino_hog_glcm,
    y_treino
)

# Previsões
previsoes_hog_glcm = modelo_ridge_hog_glcm.predict(
    X_validacao_hog_glcm
)

# Métricas
mae_hog_glcm = mean_absolute_error(
    y_validacao,
    previsoes_hog_glcm
)

rmse_hog_glcm = np.sqrt(
    mean_squared_error(
        y_validacao,
        previsoes_hog_glcm
    )
)

r2_hog_glcm = r2_score(
    y_validacao,
    previsoes_hog_glcm
)

print("Ridge + HOG + GLCM")
print(f"MAE:  {mae_hog_glcm:.2f} meses")
print(f"RMSE: {rmse_hog_glcm:.2f} meses")
print(f"R²:   {r2_hog_glcm:.4f}")

Ridge + HOG + GLCM
MAE:  30.34 meses
RMSE: 39.90 meses
R²:   0.0566


In [35]:
# Recuperar as características LBP
X_treino_lbp = np.load("/content/X_treino_lbp.npy")
X_validacao_lbp = np.load("/content/X_validacao_lbp.npy")

print("LBP recuperado!")
print("Treino:", X_treino_lbp.shape)
print("Validação:", X_validacao_lbp.shape)

LBP recuperado!
Treino: (3000, 10)
Validação: (750, 10)


In [36]:
# Combinação HOG + LBP

X_treino_hog_lbp = np.hstack([
    X_treino_hog,
    X_treino_lbp
])

X_validacao_hog_lbp = np.hstack([
    X_validacao_hog,
    X_validacao_lbp
])

print("HOG + LBP")
print("Treino:", X_treino_hog_lbp.shape)
print("Validação:", X_validacao_hog_lbp.shape)

HOG + LBP
Treino: (3000, 8110)
Validação: (750, 8110)


In [37]:
# Ridge + HOG + LBP

modelo_ridge_hog_lbp = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=10.0))
])

modelo_ridge_hog_lbp.fit(
    X_treino_hog_lbp,
    y_treino
)

previsoes_hog_lbp = modelo_ridge_hog_lbp.predict(
    X_validacao_hog_lbp
)

mae_hog_lbp = mean_absolute_error(
    y_validacao,
    previsoes_hog_lbp
)

rmse_hog_lbp = np.sqrt(
    mean_squared_error(
        y_validacao,
        previsoes_hog_lbp
    )
)

r2_hog_lbp = r2_score(
    y_validacao,
    previsoes_hog_lbp
)

print("Ridge + HOG + LBP")
print(f"MAE:  {mae_hog_lbp:.2f} meses")
print(f"RMSE: {rmse_hog_lbp:.2f} meses")
print(f"R²:   {r2_hog_lbp:.4f}")

Ridge + HOG + LBP
MAE:  29.64 meses
RMSE: 39.23 meses
R²:   0.0879


In [38]:
# Combinação dos três descritores: HOG + GLCM + LBP

X_treino_hog_glcm_lbp = np.hstack([
    X_treino_hog,
    X_treino_glcm,
    X_treino_lbp
])

X_validacao_hog_glcm_lbp = np.hstack([
    X_validacao_hog,
    X_validacao_glcm,
    X_validacao_lbp
])

print("HOG + GLCM + LBP")
print("Treino:", X_treino_hog_glcm_lbp.shape)
print("Validação:", X_validacao_hog_glcm_lbp.shape)

HOG + GLCM + LBP
Treino: (3000, 8134)
Validação: (750, 8134)


In [39]:
# Ridge + HOG + GLCM + LBP

modelo_ridge_hog_glcm_lbp = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=10.0))
])

modelo_ridge_hog_glcm_lbp.fit(
    X_treino_hog_glcm_lbp,
    y_treino
)

previsoes_hog_glcm_lbp = modelo_ridge_hog_glcm_lbp.predict(
    X_validacao_hog_glcm_lbp
)

mae_hog_glcm_lbp = mean_absolute_error(
    y_validacao,
    previsoes_hog_glcm_lbp
)

rmse_hog_glcm_lbp = np.sqrt(
    mean_squared_error(
        y_validacao,
        previsoes_hog_glcm_lbp
    )
)

r2_hog_glcm_lbp = r2_score(
    y_validacao,
    previsoes_hog_glcm_lbp
)

print("Ridge + HOG + GLCM + LBP")
print(f"MAE:  {mae_hog_glcm_lbp:.2f} meses")
print(f"RMSE: {rmse_hog_glcm_lbp:.2f} meses")
print(f"R²:   {r2_hog_glcm_lbp:.4f}")

Ridge + HOG + GLCM + LBP
MAE:  29.75 meses
RMSE: 39.14 meses
R²:   0.0922


In [40]:
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# Validação cruzada somente no conjunto de treino
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

# Pipeline para evitar vazamento durante a padronização
pipeline_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

# Valores de alpha a serem testados
param_grid_ridge = {
    "ridge__alpha": [0.1, 1.0, 10.0, 100.0]
}

# Busca do melhor hiperparâmetro usando apenas o treino
grid_ridge = GridSearchCV(
    estimator=pipeline_ridge,
    param_grid=param_grid_ridge,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid_ridge.fit(
    X_treino_hog,
    y_treino
)

print("Melhor alpha:", grid_ridge.best_params_["ridge__alpha"])
print("Melhor MAE médio na validação cruzada:",
      -grid_ridge.best_score_)

Melhor alpha: 100.0
Melhor MAE médio na validação cruzada: 27.249982833862305


In [41]:
# Avaliação do melhor Ridge encontrado pela validação cruzada

melhor_modelo_ridge_hog = grid_ridge.best_estimator_

previsoes_ridge_hog_tunado = melhor_modelo_ridge_hog.predict(
    X_validacao_hog
)

mae_ridge_hog_tunado = mean_absolute_error(
    y_validacao,
    previsoes_ridge_hog_tunado
)

rmse_ridge_hog_tunado = np.sqrt(
    mean_squared_error(
        y_validacao,
        previsoes_ridge_hog_tunado
    )
)

r2_ridge_hog_tunado = r2_score(
    y_validacao,
    previsoes_ridge_hog_tunado
)

print("Ridge + HOG — modelo tunado")
print(f"Alpha escolhido: {grid_ridge.best_params_['ridge__alpha']}")
print(f"MAE:  {mae_ridge_hog_tunado:.2f} meses")
print(f"RMSE: {rmse_ridge_hog_tunado:.2f} meses")
print(f"R²:   {r2_ridge_hog_tunado:.4f}")

# Índice da melhor configuração
melhor_idx = grid_ridge.best_index_

# Média e desvio-padrão do MAE nos 5 folds
mae_medio = -grid_ridge.cv_results_["mean_test_score"][melhor_idx]
mae_desvio = grid_ridge.cv_results_["std_test_score"][melhor_idx]

print(f"MAE médio ± desvio-padrão: {mae_medio:.2f} ± {mae_desvio:.2f} meses")

Ridge + HOG — modelo tunado
Alpha escolhido: 100.0
MAE:  26.74 meses
RMSE: 35.81 meses
R²:   0.2400
MAE médio ± desvio-padrão: 27.25 ± 0.71 meses


In [42]:
# CV + tuning do Ridge com GLCM

pipeline_ridge_glcm = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

grid_ridge_glcm = GridSearchCV(
    estimator=pipeline_ridge_glcm,
    param_grid={
        "ridge__alpha": [0.1, 1.0, 10.0, 100.0]
    },
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid_ridge_glcm.fit(
    X_treino_glcm,
    y_treino
)

print("Ridge + GLCM")
print("Melhor alpha:", grid_ridge_glcm.best_params_["ridge__alpha"])
print(
    "Melhor MAE médio no CV:",
    -grid_ridge_glcm.best_score_
)

Ridge + GLCM
Melhor alpha: 0.1
Melhor MAE médio no CV: 30.933731842041016


In [43]:
# Avaliação do Ridge + GLCM tunado

melhor_modelo_ridge_glcm = grid_ridge_glcm.best_estimator_

previsoes_ridge_glcm_tunado = melhor_modelo_ridge_glcm.predict(
    X_validacao_glcm
)

mae_ridge_glcm_tunado = mean_absolute_error(
    y_validacao,
    previsoes_ridge_glcm_tunado
)

rmse_ridge_glcm_tunado = np.sqrt(
    mean_squared_error(
        y_validacao,
        previsoes_ridge_glcm_tunado
    )
)

r2_ridge_glcm_tunado = r2_score(
    y_validacao,
    previsoes_ridge_glcm_tunado
)

print("Ridge + GLCM — modelo tunado")
print(f"Alpha escolhido: {grid_ridge_glcm.best_params_['ridge__alpha']}")
print(f"MAE:  {mae_ridge_glcm_tunado:.2f} meses")
print(f"RMSE: {rmse_ridge_glcm_tunado:.2f} meses")
print(f"R²:   {r2_ridge_glcm_tunado:.4f}")

Ridge + GLCM — modelo tunado
Alpha escolhido: 0.1
MAE:  31.35 meses
RMSE: 38.46 meses
R²:   0.1232


In [44]:
# CV + tuning do Ridge com LBP

pipeline_ridge_lbp = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

grid_ridge_lbp = GridSearchCV(
    estimator=pipeline_ridge_lbp,
    param_grid={
        "ridge__alpha": [0.1, 1.0, 10.0, 100.0]
    },
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid_ridge_lbp.fit(
    X_treino_lbp,
    y_treino
)

print("Ridge + LBP")
print("Melhor alpha:", grid_ridge_lbp.best_params_["ridge__alpha"])
print(
    "Melhor MAE médio no CV:",
    -grid_ridge_lbp.best_score_
)

Ridge + LBP
Melhor alpha: 0.1
Melhor MAE médio no CV: 30.20047264099121


In [45]:
# Avaliação do Ridge + LBP tunado

melhor_modelo_ridge_lbp = grid_ridge_lbp.best_estimator_

previsoes_ridge_lbp_tunado = melhor_modelo_ridge_lbp.predict(
    X_validacao_lbp
)

mae_ridge_lbp_tunado = mean_absolute_error(
    y_validacao,
    previsoes_ridge_lbp_tunado
)

rmse_ridge_lbp_tunado = np.sqrt(
    mean_squared_error(
        y_validacao,
        previsoes_ridge_lbp_tunado
    )
)

r2_ridge_lbp_tunado = r2_score(
    y_validacao,
    previsoes_ridge_lbp_tunado
)

print("Ridge + LBP — modelo tunado")
print(f"Alpha escolhido: {grid_ridge_lbp.best_params_['ridge__alpha']}")
print(f"MAE:  {mae_ridge_lbp_tunado:.2f} meses")
print(f"RMSE: {rmse_ridge_lbp_tunado:.2f} meses")
print(f"R²:   {r2_ridge_lbp_tunado:.4f}")

Ridge + LBP — modelo tunado
Alpha escolhido: 0.1
MAE:  30.45 meses
RMSE: 37.85 meses
R²:   0.1509


In [46]:
from sklearn.tree import DecisionTreeRegressor

# Pipeline do Decision Tree
pipeline_tree = Pipeline([
    ("tree", DecisionTreeRegressor(random_state=SEED))
])

# Hiperparâmetros para testar
param_grid_tree = {
    "tree__max_depth": [5, 10, 15, 20],
    "tree__min_samples_split": [2, 5, 10]
}

# Busca com validação cruzada
grid_tree = GridSearchCV(
    estimator=pipeline_tree,
    param_grid=param_grid_tree,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid_tree.fit(
    X_treino_hog,
    y_treino
)

print("Decision Tree + HOG")
print("Melhores parâmetros:")
print(grid_tree.best_params_)
print(
    "Melhor MAE médio no CV:",
    -grid_tree.best_score_
)

Decision Tree + HOG
Melhores parâmetros:
{'tree__max_depth': 5, 'tree__min_samples_split': 10}
Melhor MAE médio no CV: 32.23862938294427


In [47]:
# Avaliação do Decision Tree + HOG tunado

melhor_modelo_tree_hog = grid_tree.best_estimator_

previsoes_tree_hog_tunado = melhor_modelo_tree_hog.predict(
    X_validacao_hog
)

mae_tree_hog_tunado = mean_absolute_error(
    y_validacao,
    previsoes_tree_hog_tunado
)

rmse_tree_hog_tunado = np.sqrt(
    mean_squared_error(
        y_validacao,
        previsoes_tree_hog_tunado
    )
)

r2_tree_hog_tunado = r2_score(
    y_validacao,
    previsoes_tree_hog_tunado
)

print("Decision Tree + HOG — modelo tunado")
print(f"Parâmetros: {grid_tree.best_params_}")
print(f"MAE:  {mae_tree_hog_tunado:.2f} meses")
print(f"RMSE: {rmse_tree_hog_tunado:.2f} meses")
print(f"R²:   {r2_tree_hog_tunado:.4f}")

Decision Tree + HOG — modelo tunado
Parâmetros: {'tree__max_depth': 5, 'tree__min_samples_split': 10}
MAE:  31.84 meses
RMSE: 40.37 meses
R²:   0.0341


In [48]:
# Random Forest enxuto + HOG
# Configuração fixa para comparação entre modelos

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import time

inicio = time.time()

modelo_rf_rapido = RandomForestRegressor(
    n_estimators=30,
    max_depth=10,
    min_samples_split=10,
    random_state=SEED,
    n_jobs=-1
)

modelo_rf_rapido.fit(
    X_treino_hog,
    y_treino
)

previsoes_rf_rapido = modelo_rf_rapido.predict(
    X_validacao_hog
)

mae_rf_rapido = mean_absolute_error(
    y_validacao,
    previsoes_rf_rapido
)

rmse_rf_rapido = np.sqrt(
    mean_squared_error(
        y_validacao,
        previsoes_rf_rapido
    )
)

r2_rf_rapido = r2_score(
    y_validacao,
    previsoes_rf_rapido
)

tempo = (time.time() - inicio) / 60

print("Random Forest + HOG — configuração enxuta")
print(f"MAE:  {mae_rf_rapido:.2f} meses")
print(f"RMSE: {rmse_rf_rapido:.2f} meses")
print(f"R²:   {r2_rf_rapido:.4f}")
print(f"Tempo: {tempo:.2f} minutos")

Random Forest + HOG — configuração enxuta
MAE:  28.62 meses
RMSE: 34.94 meses
R²:   0.2767
Tempo: 7.14 minutos


In [49]:
# Adicionando sexo como covariável ao HOG

sexo_treino = df_treino_amostra["male"].astype(np.float32).values.reshape(-1, 1)
sexo_validacao = df_validacao_amostra["male"].astype(np.float32).values.reshape(-1, 1)

X_treino_hog_sexo = np.hstack([
    X_treino_hog,
    sexo_treino
])

X_validacao_hog_sexo = np.hstack([
    X_validacao_hog,
    sexo_validacao
])

print("HOG + sexo")
print("Treino:", X_treino_hog_sexo.shape)
print("Validação:", X_validacao_hog_sexo.shape)

HOG + sexo
Treino: (3000, 8101)
Validação: (750, 8101)


In [50]:
# Definindo a semente de reprodutibilidade

SEED = 42

print("SEED definida como:", SEED)

SEED definida como: 42


In [51]:
# Configuração da validação cruzada

from sklearn.model_selection import KFold

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

print("Validação cruzada configurada.")
print("Número de folds:", cv.n_splits)

Validação cruzada configurada.
Número de folds: 5


In [52]:
# Ridge + HOG + sexo com validação cruzada

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

pipeline_ridge_hog_sexo = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

param_grid_ridge_hog_sexo = {
    "ridge__alpha": [0.1, 1.0, 10.0, 100.0]
}

grid_ridge_hog_sexo = GridSearchCV(
    estimator=pipeline_ridge_hog_sexo,
    param_grid=param_grid_ridge_hog_sexo,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid_ridge_hog_sexo.fit(
    X_treino_hog_sexo,
    y_treino
)

print("Melhor alpha:", grid_ridge_hog_sexo.best_params_["ridge__alpha"])
print(
    "Melhor MAE médio na validação cruzada:",
    -grid_ridge_hog_sexo.best_score_
)

Melhor alpha: 100.0
Melhor MAE médio na validação cruzada: 24.982511901855467


In [53]:
# Avaliação do Ridge + HOG + sexo no conjunto de validação

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

previsoes_ridge_hog_sexo = grid_ridge_hog_sexo.predict(
    X_validacao_hog_sexo
)

mae_ridge_hog_sexo = mean_absolute_error(
    y_validacao,
    previsoes_ridge_hog_sexo
)

rmse_ridge_hog_sexo = np.sqrt(
    mean_squared_error(
        y_validacao,
        previsoes_ridge_hog_sexo
    )
)

r2_ridge_hog_sexo = r2_score(
    y_validacao,
    previsoes_ridge_hog_sexo
)

print("Ridge + HOG + sexo")
print("Melhor alpha:", grid_ridge_hog_sexo.best_params_["ridge__alpha"])
print(f"MAE: {mae_ridge_hog_sexo:.2f} meses")
print(f"RMSE: {rmse_ridge_hog_sexo:.2f} meses")
print(f"R²: {r2_ridge_hog_sexo:.4f}")

Ridge + HOG + sexo
Melhor alpha: 100.0
MAE: 25.52 meses
RMSE: 34.20 meses
R²: 0.3067


### Inclusão da variável sexo como covariável

Foi realizado um experimento adicionando a variável `male` como covariável às características HOG. O vetor de características passou de 8.100 para 8.101 dimensões.

Para o modelo Ridge, foi utilizada uma pipeline com padronização das características e validação cruzada de 5 folds, com busca do hiperparâmetro `alpha` entre 0,1, 1, 10 e 100.

O melhor valor encontrado foi `alpha = 100`, com MAE médio de 24,98 meses na validação cruzada. Na avaliação sobre o conjunto de validação, o modelo apresentou MAE de 25,52 meses, RMSE de 34,20 meses e R² de 0,3067.

A comparação com o modelo Ridge + HOG, sem a covariável sexo, apresentou diferenças nas métricas de validação, permitindo avaliar experimentalmente a contribuição dessa informação adicional.